### Importation des bibliothèques

Les bibliothèques nécessaires sont importées afin de manipuler les données et réaliser les visualisations.

In [ ]:
%pip install numpy pandas pandas seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

### Chargement des données

Le fichier `mesures_capteurs.csv` est importé dans un DataFrame nommé `df`. Ce DataFrame sera utilisé tout au long de l'atelier.

In [ ]:
df = pd.read_csv("../data/mesures_capteurs.csv")

df

### Exploration des données

Une première inspection du DataFrame permet de vérifier que les données ont été correctement chargées et d'observer leur structure.

In [ ]:
df.head()

df.info()

df.describe()

df.isnull().sum()

### 1.1 : Vérification des doublons

Avant de poursuivre l'analyse des données, il est important de vérifier si le DataFrame contient des lignes dupliquées. Les doublons peuvent fausser les statistiques et les visualisations réalisées par la suite.

In [ ]:
# Nombre de lignes dupliquées
nb_doublons = df.duplicated().sum()

print(f"Nombre de doublons : {nb_doublons}")

### 1.2 : Suppression des doublons

Si des doublons sont présents, ils sont supprimés afin de conserver une seule occurrence de chaque observation. Une nouvelle vérification est ensuite réalisée pour confirmer que le nettoyage a bien été effectué.

In [ ]:
# Suppression des doublons
df = df.drop_duplicates()

# Vérification après suppression
nb_doublons = df.duplicated().sum()

print(f"Nombre de doublons après suppression : {nb_doublons}")

### 2.1 : Définition de la cible et des caractéristiques

Dans cette étape, la colonne `etat` est définie comme la variable cible (`y`), c'est-à-dire la valeur que le modèle devra prédire.

Les colonnes `temperature`, `humidite`, `pression` et `consommation` constituent les variables explicatives (`X`) qui serviront à entraîner le modèle.

In [ ]:
# Variable cible
y = df["etat"]

# Variables explicatives
X = df[["temperature", "humidite", "pression", "consommation"]]

### 2.2 : Aperçu des données

Les cinq premières lignes de `X` et de `y` sont affichées afin de vérifier que les variables ont été correctement sélectionnées.

In [ ]:
print("Variables explicatives (X) :")
display(X.head())

print("Variable cible (y) :")
display(y.head())

### 2.3 : Type du problème

La variable cible `etat` contient des catégories (par exemple : `OK`, `ALERTE` et `ERREUR`). Le modèle devra prédire une classe à partir de plusieurs caractéristiques.

Il s'agit donc d'un problème de **classification**.

In [ ]:
print("Type du problème :", "Classification")

print("\nClasses présentes dans la cible :")
print(y.unique())

### 3.1 : Découpage Train/Test

Les données sont divisées en deux ensembles :

- un ensemble d'entraînement (`train`) utilisé pour apprendre le modèle ;
- un ensemble de test (`test`) utilisé pour évaluer ses performances.

Le découpage est réalisé en réservant 20 % des données au test. La reproductibilité est assurée grâce à `random_state`, tandis que `stratify` permet de conserver la même répartition des classes dans les deux ensembles.

In [ ]:
# Vérifier les stats de NaN

df.isnull().sum()

In [ ]:
# suppresion des valeurs manquantes

df = df.dropna()

# Redéfinir X et y
X = df[["temperature", "humidite", "pression", "consommation"]]
y = df["etat"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### 3.2 : Vérifier du découpage

Les dimensions des ensembles obtenus sont affichées afin de vérifier que le découpage a été correctement réalisé.

In [ ]:
print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test  :", X_test.shape)

print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test  :", y_test.shape)

### 4.1 : Vérifier des valeurs manquantes

Avant d'entraîner un modèle de Machine Learning, il est important de vérifier si les variables explicatives contiennent des valeurs manquantes. Ces valeurs devront être traitées afin d'éviter des erreurs lors de l'entraînement du modèle.

In [ ]:
# Nombre de valeurs manquantes par variable
X_train.isnull().sum()

In [ ]:
print("Valeurs manquantes dans X_train :")
print(X_train.isnull().sum())

print("\nValeurs manquantes dans X_test :")
print(X_test.isnull().sum())

### 4.2 : Sélection de l'imputeur

L'imputeur `SimpleImputer` est utilisé pour remplacer automatiquement les valeurs manquantes. La stratégie choisie est la médiane.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

### 4.3 : Choix de la médiane

La médiane est moins sensible aux valeurs extrêmes que la moyenne. Elle permet donc de remplacer les valeurs manquantes sans être fortement influencée par d'éventuelles observations atypiques présentes dans les données.

### 4.4 : Apprentissage de l'imputeur

L'imputeur est ajusté uniquement sur les données d'entraînement (`X_train`). Il calcule la médiane de chaque variable afin de remplacer les valeurs manquantes.

In [ ]:
imputer.fit(X_train)

# Médianes calculées
print(imputer.statistics_)

### 4.5 : Remplacement des valeurs manquantes

Les ensembles d'entraînement et de test sont transformés à l'aide de l'imputeur. Les valeurs manquantes sont remplacées par les médianes calculées sur `X_train`.

In [ ]:
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print(X_train_imputed, X_test_imputed)

### 5.1 : Mise à l'échelle avec StandardScaler

Les variables de notre jeu de données n'ont pas nécessairement les mêmes unités ni les mêmes ordres de grandeur.

Nous utilisons `StandardScaler` de Scikit-learn pour standardiser les données après l'imputation des valeurs manquantes.

La standardisation transforme chaque variable selon la formule :

$$z = \frac{x - \mu}{\sigma}$$

où :
- $\mu$ représente la moyenne de la variable ;
- $\sigma$ représente son écart-type.

Le `StandardScaler` sera ajusté uniquement sur `X_train_imputed`, puis utilisé pour transformer les données d'entraînement et de test.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

### 5.2 : Pourquoi standardiser les données ?

La standardisation est importante pour le modèle KNN car celui-ci utilise les distances entre les observations pour déterminer les voisins les plus proches.

Si une variable possède des valeurs beaucoup plus grandes qu'une autre, elle peut avoir une influence disproportionnée sur le calcul des distances.

La standardisation permet donc de placer les différentes variables sur une échelle comparable, avec une moyenne proche de 0 et un écart-type proche de 1.

Cela permet au modèle KNN de prendre en compte les caractéristiques de manière plus équilibrée.

In [ ]:
print("StandardScaler sélectionné pour la mise à l'échelle des données.")

### 5.3 : Paramètres du StandardScaler

Le `StandardScaler` doit être ajusté uniquement sur les données d'entraînement.

La méthode `fit()` permet de calculer, pour chaque variable, les paramètres nécessaires à la standardisation :

- la moyenne ;
- l'écart-type.

Ces paramètres seront ensuite utilisés pour transformer les données d'entraînement et de test.

In [ ]:
scaler.fit(X_train_imputed)

print("Moyennes :")
print(scaler.mean_)

print("\nÉcarts-types :")
print(scaler.scale_)

### 5.4 : Transformation des données

Le `StandardScaler` étant maintenant ajusté sur `X_train_imputed`, nous pouvons transformer les données d'entraînement et de test.

La méthode `transform()` utilise les paramètres calculés à partir de `X_train_imputed`.

Nous obtenons ainsi :

- `X_train_scaled` : les données d'entraînement standardisées ;
- `X_test_scaled` : les données de test standardisées.

Les paramètres ne sont donc pas recalculés sur les données de test.

In [ ]:
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("X_train_scaled :")
print(X_train_scaled[:5])

print("\nX_test_scaled :")
print(X_test_scaled[:5])

### 6.1 : Sélection du modèle KNN

Le modèle **K-Nearest Neighbors (KNN)** est choisi pour réaliser la classification des états des capteurs.

Le principe de KNN consiste à rechercher les **k voisins les plus proches** d'une nouvelle observation, puis à lui attribuer la classe la plus représentée parmi ces voisins.

Dans cet atelier, nous choisissons **k = 5**, ce qui signifie que les cinq voisins les plus proches seront pris en compte pour effectuer la prédiction.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

### 6.2 : Entraînement du modèle

Le modèle est entraîné à l'aide des données d'entraînement standardisées (`X_train_scaled`) et des étiquettes correspondantes (`y_train`).

La méthode `fit()` permet au modèle d'apprendre les relations entre les caractéristiques et les différentes classes.

In [48]:
knn.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[object](3,)","['ALERTE','ERREUR','OK']"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


### 6.3 : Prédiction avec le modèle

Une fois entraîné, le modèle est utilisé pour prédire l'état des capteurs présents dans l'ensemble de test.

Les prédictions sont stockées dans la variable `y_pred`.

In [47]:
y_pred = knn.predict(X_test_scaled)

print(y_pred)

['OK' 'OK' 'ALERTE' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK'
 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK'
 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK'
 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK'
 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK'
 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'ALERTE' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK'
 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK' 'OK'
 'OK' 'OK' 'OK' 'OK' 'ALERTE' 'OK' 'OK' 'OK' 'OK' 'ALERTE' 'OK' 'OK' 'OK'
 'OK' 'OK' 'OK' 'OK']


### 6.4 : Aperçu des prédictions

Quelques prédictions sont affichées afin d'observer les résultats produits par le modèle sur les premières observations de l'ensemble de test.

In [ ]:
print("Quelques prédictions :")

for i, prediction in enumerate(y_pred[:10], start=1):
    print(f"Observation {i} : {prediction}")

### 6.5 : Comparaison des prédictions

Les prédictions du modèle sont comparées aux véritables valeurs de l'ensemble de test.

Cette comparaison permet d'observer, pour chaque observation, si le modèle a correctement identifié l'état du capteur.

In [ ]:
comparaison = pd.DataFrame({
    "Valeur réelle": y_test.values,
    "Prédiction": y_pred
})

comparaison.head(10)

### 7.1 : Évaluation avec l'Accuracy

L'**Accuracy** (exactitude) mesure la proportion de prédictions correctes réalisées par le modèle.

Elle est calculée en comparant les prédictions (`y_pred`) aux vraies valeurs (`y_test`).

Sa valeur est comprise entre **0** et **1**. Plus elle est proche de **1**, plus le modèle est performant.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")

### 7.2 : Pourquoi l'Accuracy ne suffit-elle pas ?

L'Accuracy peut être trompeuse lorsque les classes sont déséquilibrées.

Par exemple, si 95 % des observations appartiennent à une même classe, un modèle qui prédit toujours cette classe obtiendra une Accuracy élevée, tout en étant incapable de détecter correctement les autres classes.

Il est donc nécessaire de compléter l'évaluation avec d'autres indicateurs tels que la précision, le rappel et le F1-score.

### 7.3 : Matrice de confusion

La matrice de confusion compare les classes réelles aux classes prédites.

Chaque ligne représente les valeurs réelles tandis que chaque colonne représente les prédictions du modèle.

Une matrice proche de la diagonale indique généralement de bonnes performances.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Matrice de confusion :")
print(cm)

### Commentaire

La matrice de confusion montre que le modèle identifie très bien la classe **OK**, avec 108 prédictions correctes sur 108.

En revanche, les classes **ALERTE** et **ERREUR** sont plus difficiles à reconnaître. La moitié des observations **ALERTE** est confondue avec la classe **OK**, tandis que l'unique observation **ERREUR** est classée comme **ALERTE**.

Ces résultats suggèrent un déséquilibre entre les classes, ce qui favorise la prédiction de la classe majoritaire (**OK**).

### 7.4 : Visualisation avec Seaborn

La matrice de confusion est représentée sous forme de carte thermique (heatmap) afin de faciliter son interprétation.

Les valeurs situées sur la diagonale correspondent aux prédictions correctes.

In [ ]:
plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=knn.classes_,
    yticklabels=knn.classes_
)

plt.title("Matrice de confusion")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")

plt.show()

### Commentaire

La représentation graphique confirme que la majorité des prédictions correctes se situe sur la diagonale principale, en particulier pour la classe **OK**.

Les erreurs de classification concernent principalement les classes **ALERTE** et **ERREUR**, qui disposent de beaucoup moins d'exemples dans le jeu de données.

Le modèle obtient donc d'excellentes performances sur la classe dominante, mais éprouve davantage de difficultés à distinguer les classes minoritaires.

### 7.5 : Rapport de classification

Le rapport de classification fournit plusieurs indicateurs de performance pour chaque classe :

- la précision (*Precision*) ;
- le rappel (*Recall*) ;
- le F1-score ;
- le nombre d'observations (*Support*).

Ces indicateurs permettent d'obtenir une évaluation plus détaillée que la seule Accuracy.

In [ ]:
from sklearn.metrics import classification_report

rapport = classification_report(y_test, y_pred)

print("Rapport de classification :")
print(rapport)

### Commentaire

Le rapport de classification montre que le modèle obtient une **Accuracy de 97 %**, ce qui indique de très bonnes performances globales.

La classe **OK** est très bien reconnue, avec une précision de **0,97**, un rappel de **1,00** et un F1-score de **0,99**.

La classe **ALERTE** est correctement détectée dans certains cas, mais seulement **50 %** des observations sont reconnues, ce qui se traduit par un rappel de **0,50**.

En revanche, la classe **ERREUR** n'est jamais correctement prédite. Sa précision, son rappel et son F1-score sont tous égaux à **0,00**.

Ces résultats montrent que le modèle est fortement influencé par le déséquilibre des classes : la classe **OK** est largement majoritaire, tandis que **ALERTE** et surtout **ERREUR** sont très peu représentées. Il est donc important de ne pas se fier uniquement à l'Accuracy et d'analyser également les autres métriques.

### 7.6 : Quand utiliser chaque indicateur ?

Le choix de la métrique dépend du problème étudié.

- **Precision** : à privilégier lorsque les faux positifs sont coûteux. Exemple : détection de spam, où il est important de ne pas classer un e-mail légitime comme spam.

- **Recall** : à privilégier lorsque les faux négatifs sont les plus critiques. Exemple : diagnostic médical, où il est essentiel de détecter un maximum de cas positifs.

- **F1-score** : à utiliser lorsque l'on souhaite trouver un équilibre entre la précision et le rappel, notamment avec des classes déséquilibrées.

- **Accuracy** : adaptée lorsque les classes sont relativement équilibrées et que toutes les erreurs ont une importance comparable.